# Build a Gauss–Hermite filter with Cuthbert

A nonlinear transition can bend a Gaussian into a non-Gaussian distribution. Here we propagate a small set of weighted points through the transition and observation functions, then approximate each transformed distribution by its first two moments. We implement the recursion as a Cuthbert filter and let Dynestyx handle observation times, output collection, and NumPyro registration.

In [ ]:
from typing import NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import numpyro.distributions as dist
from numpyro.handlers import seed, trace
from cuthbert.inference import Filter
from cuthbertlib.quadrature import gauss_hermite

import dynestyx as dsx
from dynestyx.configs import CustomFilterConfig, KFConfig
from dynestyx.inference.custom_filter import FilterOutputs
from dynestyx.inference.observation_predictions import PredictedObservationOutputs

## Transform a Gaussian

Order-three Gauss–Hermite quadrature uses three points per state dimension, so the cost grows as $3^d$. For each point, the model supplies a conditional mean and covariance. The covariance of the transformed means captures variation in the signal. Adding the average conditional covariance accounts for process or measurement noise. The cross-covariance records how the input state and transformed signal vary together.

In [ ]:
def transform(rule, mean, cov, conditional):
    points = rule.get_sigma_points(mean, jnp.linalg.cholesky(cov))

    def moments(x):
        distribution = conditional(x)
        return distribution.mean, distribution.covariance_matrix

    means, covs = jax.vmap(moments)(points.points)
    signals = points._replace(points=means)
    noise = jnp.einsum("i,ijk->jk", points.wm, covs)
    return signals.mean, signals.covariance(), noise, points.covariance(signals)


class State(NamedTuple):
    mean: jax.Array
    cov: jax.Array
    loglik: jax.Array
    predictions: PredictedObservationOutputs
    innovation: jax.Array

## Bind the filter to a model

The initial Gaussian is the prior at the first observation. Later observations require a transition from the previous time using its control. For the update, solve against the noisy observation covariance to obtain the gain, then score the innovation under that same predictive Gaussian.

In [ ]:
def build_filter(model):
    rule = gauss_hermite.weights(model.state_dim, order=3)

    def initialize(*, key):
        prior = model.initial_condition
        zero = jnp.zeros((model.observation_dim,), dtype=prior.mean.dtype)
        zeros = jnp.outer(zero, zero)
        predictions = PredictedObservationOutputs(mean=zero, cov=zeros, obs_cov=zeros)
        return State(prior.mean, prior.covariance_matrix, jnp.zeros_like(zero[0]),
                     predictions, zero)

    def prepare(inputs, *, key):
        return inputs

    def combine(state, inputs):
        def predict():
            transition = lambda x: model.state_evolution(
                x, inputs.u_prev, inputs.time_prev, inputs.time
            )
            m, signal, noise, _ = transform(rule, state.mean, state.cov, transition)
            return m, signal + noise

        mean, cov = jax.lax.cond(
            inputs.is_first_step, lambda: (state.mean, state.cov), predict
        )
        observe = lambda x: model.observation_model(x, inputs.u, inputs.time)
        y_mean, signal, noise, cross = transform(rule, mean, cov, observe)
        y_cov = signal + noise
        gain = jnp.linalg.solve(y_cov, cross.T).T
        innovation = inputs.y - y_mean
        loglik = dist.MultivariateNormal(y_mean, y_cov).log_prob(inputs.y)
        return State(
            mean + gain @ innovation, cov - gain @ cross.T, state.loglik + loglik,
            PredictedObservationOutputs(mean=y_mean, cov=signal, obs_cov=y_cov),
            innovation,
        )

    return Filter(initialize, prepare, combine, associative=False)


def gather_outputs(state):
    return FilterOutputs(
        state.loglik, state.mean, state.cov, predicted_observations=state.predictions,
        extra_outputs={"innovation": state.innovation},
    )


config = CustomFilterConfig(build_filter=build_filter, gather_outputs=gather_outputs)

## Condition and score

The scalar latent state has a saturating transition and a mildly curved observation function. The recorded innovation is the observed value minus its predicted mean, before assimilation.

In [ ]:
model = dsx.DynamicalModel(
    control_dim=0,
    initial_condition=dist.MultivariateNormal(jnp.zeros(1), jnp.eye(1)),
    state_evolution=dsx.GaussianStateEvolution(
        F=lambda x, u, t, t_next: jnp.tanh(x), cov=0.1 * jnp.eye(1)
    ),
    observation_model=dsx.GaussianObservation(
        h=lambda x, u, t: x + 0.15 * x**2, R=0.2 * jnp.eye(1)
    ),
)
data = dict(obs_times=jnp.arange(4.0), obs_values=jnp.array([[0.3], [0.1], [-0.4], [0.2]]))
with dsx.Filter(config):
    result = dsx.condition("x", model, **data)
np.testing.assert_allclose(result.extra_outputs["innovation"],
                           data["obs_values"] - result.predicted_observations.mean)
with trace() as sites, seed(rng_seed=0), dsx.Filter(config):
    dsx.sample("x", model, **data)
np.testing.assert_allclose(sites["x_marginal_log_likelihood"]["fn"].log_factor,
                           result.marginal_loglik)
result.extra_outputs["innovation"]

## Check the linear limit

Quadrature integrates linear functions and their second moments exactly. Replacing the two nonlinear maps with linear ones should recover the Kalman filter at every observation, including the first.

In [ ]:
linear = dsx.LTI_discrete(A=jnp.array([[0.8]]), Q=0.1*jnp.eye(1),
                          H=jnp.eye(1), R=0.2*jnp.eye(1))
with dsx.Filter(config):
    gh = dsx.condition("x", linear, **data)
with dsx.Filter(KFConfig(filter_source="cuthbert")):
    exact = dsx.condition("x", linear, **data)
for a, b in zip(gh.dists, exact.dists, strict=True):
    np.testing.assert_allclose(a.mean, b.mean, atol=1e-5)
    np.testing.assert_allclose(a.covariance_matrix, b.covariance_matrix, atol=1e-5)
np.testing.assert_allclose(gh.log_normalizing_constant, exact.log_normalizing_constant, atol=1e-5)
np.testing.assert_allclose(gh.marginal_loglik, exact.marginal_loglik, atol=1e-5)